# Testes Diagnosticos para Modelos GARCH

Apos estimar um modelo GARCH, e essencial verificar se o modelo esta **bem especificado**.
Os testes diagnosticos avaliam se os residuos padronizados se comportam como ruido branco
e se a estrutura de volatilidade foi adequadamente capturada.

**Neste notebook, cobrimos:**

1. **ARCH-LM** (Engle, 1982) — testa efeitos ARCH residuais
2. **Ljung-Box** — testa autocorrelacao nos residuos padronizados
3. **Sign Bias** (Engle & Ng, 1993) — testa assimetria nao capturada
4. **Nyblom** (1989) — testa estabilidade dos parametros
5. **Diagnostico visual** — QQ-plot, ACF, PACF
6. **Workflow completo** — funcao que resume todos os testes

**Regra geral:** se o modelo esta bem especificado, os residuos padronizados
$z_t = \varepsilon_t / \sigma_t$ devem ser i.i.d. com media zero e variancia unitaria.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from archbox.diagnostics import (
    arch_lm_test,
    full_diagnostics,
    ljung_box_squared,
    nyblom_test,
    sign_bias_test,
)
from archbox.models import EGARCH, GARCH, GJRGARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns'].values

# Carregar dados com assimetria (gerados por EGARCH)
data_asym = pd.read_csv('../data/misspecified_returns.csv', parse_dates=['date'], index_col='date')
returns_asym = data_asym['returns'].values

# Carregar dados com quebra estrutural
data_break = pd.read_csv('../data/structural_break_returns.csv', parse_dates=['date'], index_col='date')
returns_break = data_break['returns'].values

print(f"Dataset SP500: {len(returns)} obs")
print(f"Dataset assimetrico (EGARCH DGP): {len(returns_asym)} obs")
print(f"Dataset com quebra estrutural: {len(returns_break)} obs")

## 1. Estimando o modelo base

Vamos comecar estimando um **GARCH(1,1)** simetrico nos dados do S&P 500.
Este sera nosso modelo base para os diagnosticos.

Apos a estimacao, extraimos os **residuos padronizados**:

$$z_t = \frac{\varepsilon_t}{\sigma_t}$$

Se o modelo esta correto, $z_t \sim \text{i.i.d.}(0, 1)$.

In [ ]:
# TODO: Estime GARCH(1,1) e extraia residuos padronizados
# 1. Estime o modelo nos dados SP500
# 2. Extraia residuos padronizados (z_t) e volatilidade condicional (sigma_t)
# 3. Calcule tambem os residuos "raw" (eps_t = returns demeaned)

# Modelo GARCH(1,1)
model_garch = GARCH(returns, p=1, q=1)
results_garch = model_garch.fit(disp=False)
print(results_garch.summary())

# Residuos padronizados e raw
std_resids = results_garch.resid          # z_t = eps_t / sigma_t
sigma_t = results_garch.conditional_volatility  # sigma_t
raw_resids = model_garch.endog            # eps_t (retornos demeaned)

print(f"\nResiduos padronizados: media={std_resids.mean():.4f}, var={std_resids.var():.4f}")
print(f"Persistencia: {results_garch.persistence():.4f}")
print(f"Meia-vida: {results_garch.half_life():.1f} dias")

## 2. Teste ARCH-LM de Engle (1982)

O teste **ARCH-LM** verifica se restam efeitos ARCH nos residuos padronizados.
Se o modelo GARCH capturou toda a dependencia na volatilidade, os residuos
padronizados ao quadrado $z_t^2$ nao devem ter autocorrelacao.

**Hipoteses:**
- $H_0$: Nao ha efeitos ARCH residuais (homocedasticidade em $z_t^2$)
- $H_1$: Ha efeitos ARCH residuais

**Regressao auxiliar:**

$$z_t^2 = \alpha_0 + \alpha_1 z_{t-1}^2 + \cdots + \alpha_q z_{t-q}^2 + v_t$$

**Estatistica:** $LM = T \times R^2 \sim \chi^2(q)$

Se $p < 0.05$: rejeita $H_0$ → modelo mal especificado (efeitos ARCH residuais).

In [ ]:
# TODO: Execute ARCH-LM test nos residuos com lags 1, 5, 10
# Se o GARCH(1,1) esta bem especificado, todos devem ser nao-significativos

print("ARCH-LM Test nos residuos padronizados do GARCH(1,1)")
print("=" * 55)
print(f"{'Lags':<8} {'Estatistica':>12} {'p-valor':>12} {'Decisao':>12}")
print("-" * 55)

for lags in [1, 5, 10]:
    result = arch_lm_test(std_resids, lags=lags)
    decision = "PASS (H0)" if result.pvalue > 0.05 else "FAIL (H1)"
    print(f"{lags:<8} {result.statistic:>12.4f} {result.pvalue:>12.6f} {decision:>12}")

print("\nInterpretacao:")
print("PASS = nao rejeita H0 (sem efeitos ARCH residuais) → modelo adequado")
print("FAIL = rejeita H0 (efeitos ARCH residuais) → modelo insuficiente")

## 3. Teste de Ljung-Box

O teste de **Ljung-Box** verifica se ha autocorrelacao nos residuos padronizados
(ou nos residuos ao quadrado). E um teste mais geral que o ARCH-LM.

**Hipoteses:**
- $H_0$: Nao ha autocorrelacao ate a defasagem $m$ (residuos sao ruido branco)
- $H_1$: Ha autocorrelacao em pelo menos uma defasagem

**Estatistica:**

$$Q(m) = T(T+2) \sum_{k=1}^{m} \frac{\hat{\rho}_k^2}{T-k} \sim \chi^2(m)$$

Aplicamos em **dois niveis**:
- Nos residuos padronizados $z_t$ → testa autocorrelacao na media
- Nos residuos ao quadrado $z_t^2$ → testa autocorrelacao na variancia

In [ ]:
# TODO: Execute Ljung-Box nos residuos e residuos quadrados
# ljung_box_squared testa autocorrelacao nos residuos ao quadrado (z_t^2)

print("Ljung-Box Test nos residuos padronizados ao quadrado (z_t^2)")
print("=" * 55)
print(f"{'Lags':<8} {'Q-stat':>12} {'p-valor':>12} {'Decisao':>12}")
print("-" * 55)

for lags in [5, 10, 20]:
    result = ljung_box_squared(std_resids, lags=lags)
    decision = "PASS (H0)" if result.pvalue > 0.05 else "FAIL (H1)"
    print(f"{lags:<8} {result.statistic:>12.4f} {result.pvalue:>12.6f} {decision:>12}")

print("\nInterpretacao:")
print("PASS = nao rejeita H0 → sem autocorrelacao residual em z_t^2")
print("FAIL = rejeita H0 → modelo nao capturou toda a dependencia na volatilidade")

## 4. Sign Bias Test de Engle-Ng (1993)

O **Sign Bias Test** verifica se choques positivos e negativos afetam a volatilidade
de forma diferente — algo que um modelo GARCH simetrico **nao captura**.

Sao **3 componentes** + teste conjunto:

**Regressao:**

$$z_t^2 = c_0 + c_1 S_{t-1}^- + c_2 S_{t-1}^- \varepsilon_{t-1} + c_3 S_{t-1}^+ \varepsilon_{t-1} + u_t$$

onde $S_{t-1}^- = \mathbf{1}(\varepsilon_{t-1} < 0)$ e $S_{t-1}^+ = \mathbf{1}(\varepsilon_{t-1} \geq 0)$.

| Componente | Hipotese $H_0$ | O que detecta |
|---|---|---|
| **Sign Bias** ($c_1$) | $c_1 = 0$ | Choques negativos vs positivos tem impacto diferente no nivel |
| **Negative Size Bias** ($c_2$) | $c_2 = 0$ | A magnitude dos choques negativos importa |
| **Positive Size Bias** ($c_3$) | $c_3 = 0$ | A magnitude dos choques positivos importa |
| **Joint Test** | $c_1 = c_2 = c_3 = 0$ | Qualquer tipo de assimetria presente |

In [ ]:
# TODO: Execute sign bias test completo (3 componentes + joint)
# Primeiro, no modelo GARCH simetrico com dados assimetricos (EGARCH DGP)

# Estimar GARCH(1,1) nos dados gerados por EGARCH (mal especificado!)
model_asym = GARCH(returns_asym, p=1, q=1)
results_asym = model_asym.fit(disp=False)

std_resids_asym = results_asym.resid
raw_resids_asym = model_asym.endog

# Sign bias test
sb_result = sign_bias_test(raw_resids_asym, std_resids_asym)

print("Sign Bias Test — GARCH(1,1) em dados assimetricos (EGARCH DGP)")
print("=" * 60)
print(f"{'Componente':<22} {'Estatistica':>12} {'p-valor':>12} {'Sig. 5%':>10}")
print("-" * 60)

components = [
    ("Sign Bias", sb_result.sign_bias),
    ("Neg. Size Bias", sb_result.neg_sign_bias),
    ("Pos. Size Bias", sb_result.pos_sign_bias),
]

for name, (stat, pval) in components:
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
    print(f"{name:<22} {stat:>12.4f} {pval:>12.6f} {sig:>10}")

f_stat, f_pval = sb_result.joint
sig = "***" if f_pval < 0.01 else "**" if f_pval < 0.05 else "*" if f_pval < 0.10 else ""
print(f"{'Joint Test (F)':<22} {f_stat:>12.4f} {f_pval:>12.6f} {sig:>10}")

print("\n*** p<0.01, ** p<0.05, * p<0.10")
print("\nSe joint test e significativo → modelo simetrico e INADEQUADO!")

## 5. Interpretando o sign bias

Se o sign bias test e significativo ao estimar um GARCH simetrico, isso indica que
o modelo nao captura a **assimetria** na volatilidade (efeito alavancagem).

**Solucao:** estimar um modelo assimetrico como **EGARCH** ou **GJR-GARCH**.
Apos re-estimacao, o sign bias test deve melhorar (tornar-se nao significativo).

Vamos verificar: estimamos um EGARCH nos mesmos dados e repetimos o teste.

In [ ]:
# TODO: Estime EGARCH e repita o sign bias test (deve melhorar)
# Comparacao: GARCH simetrico vs EGARCH assimetrico nos dados assimetricos

model_egarch = EGARCH(returns_asym, p=1, q=1)
results_egarch = model_egarch.fit(disp=False)

std_resids_egarch = results_egarch.resid
raw_resids_egarch = model_egarch.endog

sb_egarch = sign_bias_test(raw_resids_egarch, std_resids_egarch)

print("Comparacao: Sign Bias Test")
print("=" * 65)

# GARCH simetrico
print("\n--- GARCH(1,1) simetrico (MAL ESPECIFICADO) ---")
print(f"  Sign Bias:     t={sb_result.sign_bias[0]:.4f}, p={sb_result.sign_bias[1]:.4f}")
print(f"  Neg Size Bias: t={sb_result.neg_sign_bias[0]:.4f}, p={sb_result.neg_sign_bias[1]:.4f}")
print(f"  Pos Size Bias: t={sb_result.pos_sign_bias[0]:.4f}, p={sb_result.pos_sign_bias[1]:.4f}")
print(f"  Joint (F):     F={sb_result.joint[0]:.4f}, p={sb_result.joint[1]:.4f}")

# EGARCH assimetrico
print("\n--- EGARCH(1,1) assimetrico (CORRETO) ---")
print(f"  Sign Bias:     t={sb_egarch.sign_bias[0]:.4f}, p={sb_egarch.sign_bias[1]:.4f}")
print(f"  Neg Size Bias: t={sb_egarch.neg_sign_bias[0]:.4f}, p={sb_egarch.neg_sign_bias[1]:.4f}")
print(f"  Pos Size Bias: t={sb_egarch.pos_sign_bias[0]:.4f}, p={sb_egarch.pos_sign_bias[1]:.4f}")
print(f"  Joint (F):     F={sb_egarch.joint[0]:.4f}, p={sb_egarch.joint[1]:.4f}")

print("\nO EGARCH deve mostrar p-valores maiores (nao significativos)")
print("porque captura a assimetria via parametro gamma.")

## 6. Nyblom Stability Test (1989)

O teste de **Nyblom** verifica se os parametros do modelo sao **constantes ao longo do tempo**.
Se houve uma **quebra estrutural** (ex: mudanca de regime de volatilidade), os parametros
estimados com a amostra completa serao uma media ponderada dos verdadeiros parametros
de cada regime.

**Hipoteses:**
- $H_0$: Os parametros sao constantes ao longo do tempo ($\theta_t = \theta \ \forall t$)
- $H_1$: Os parametros variam ao longo do tempo (instabilidade parametrica)

**Estatistica conjunta:**

$$L_c = \frac{1}{T^2} \sum_{t=1}^{T} S_t' V^{-1} S_t$$

onde $S_t = \sum_{s=1}^{t} g_s$ e a soma acumulada dos scores e $V = \frac{1}{T} \sum_t g_t g_t'$.

**Estatistica individual:** $L_i = \frac{1}{T^2} \sum_t S_{i,t}^2 / V_{ii}$ para cada parametro.

Os valores criticos sao tabelados (Nyblom, 1989; Hansen, 1990).

In [ ]:
# TODO: Execute Nyblom test para cada parametro e joint
# Usamos dados com quebra estrutural para demonstrar o teste
# O score matrix precisa ser construido via diferenciacao numerica do log-likelihood

# Estimar GARCH(1,1) nos dados com quebra estrutural
model_break = GARCH(returns_break, p=1, q=1)
results_break = model_break.fit(disp=False)
print("GARCH(1,1) estimado nos dados com quebra estrutural:")
print(results_break.summary())

# Construir score matrix via diferenciacao numerica do log-likelihood por observacao
def compute_score_matrix(model, results, eps=1e-5):
    """Compute score matrix via numerical differentiation of per-obs log-likelihood."""
    params = results.params.copy()
    n_params = len(params)

    scores_list = []
    for j in range(n_params):
        params_up = params.copy()
        params_down = params.copy()
        params_up[j] += eps
        params_down[j] -= eps

        ll_up = model.loglike_per_obs(params_up)
        ll_down = model.loglike_per_obs(params_down)

        scores_list.append((ll_up - ll_down) / (2 * eps))

    return np.column_stack(scores_list)

scores_break = compute_score_matrix(model_break, results_break)
nyblom_result = nyblom_test(scores_break)

print("\n" + "=" * 60)
print("Nyblom Stability Test — dados com quebra estrutural")
print("=" * 60)
print(f"\nJoint statistic: {nyblom_result.joint_statistic:.4f}")
cv = nyblom_result.critical_values_joint
print(f"Critical values: 10%={cv[0]:.3f}, 5%={cv[1]:.3f}, 1%={cv[2]:.3f}")
print(f"Rejeita H0 a 5%: {nyblom_result.joint_rejects_5pct}")

print("\nIndividual statistics:")
cv_ind = nyblom_result.critical_values_individual
param_names = results_break.param_names
for _i, (name, stat) in enumerate(zip(param_names, nyblom_result.individual_statistics, strict=False)):
    rejects = "INSTAVEL" if stat > cv_ind[1] else "estavel"
    print(f"  {name:<10}: L_i = {stat:.4f}  (cv5% = {cv_ind[1]:.3f}) → {rejects}")

print("\nInterpretacao: se rejeita H0, os parametros NAO sao constantes")
print("(indica quebra estrutural ou mudanca de regime)")

## 7. Diagnostico visual: QQ-plot e ACF

Alem dos testes formais, graficos diagnosticos fornecem informacao complementar:

- **QQ-plot**: compara a distribuicao empirica dos residuos com a distribuicao teorica (normal).
  Desvios nas caudas indicam caudas pesadas.
- **ACF dos residuos padronizados**: nao deve haver autocorrelacao significativa.
- **ACF dos residuos ao quadrado**: idem — se o modelo capturou a volatilidade.
- **Serie temporal dos residuos**: deve parecer ruido branco.

In [ ]:
# TODO: Plote painel diagnostico completo (residuos, ACF, PACF, QQ)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Painel 1: Serie de residuos padronizados
axes[0, 0].plot(std_resids, linewidth=0.5, color='steelblue')
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuos Padronizados $z_t$')
axes[0, 0].set_xlabel('Observacao')
axes[0, 0].set_ylabel('$z_t$')

# Painel 2: ACF dos residuos ao quadrado
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(std_resids**2, lags=30, ax=axes[0, 1], title='ACF de $z_t^2$')
axes[0, 1].set_xlabel('Defasagem')

# Painel 3: PACF dos residuos ao quadrado
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(std_resids**2, lags=30, ax=axes[1, 0], title='PACF de $z_t^2$')
axes[1, 0].set_xlabel('Defasagem')

# Painel 4: QQ-plot
stats.probplot(std_resids, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('QQ-Plot (Normal)')
axes[1, 1].get_lines()[0].set_markersize(2)

fig.suptitle('Diagnostico Visual — GARCH(1,1) em S&P 500', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Interpretacao:")
print("- ACF/PACF de z_t^2 sem barras significativas → modelo captura a volatilidade")
print("- QQ-plot com desvios nas caudas → distribuicao normal pode ser inadequada")
print("  (considerar Student-t ou GED)")

## 8. Workflow diagnostico completo

Na pratica, queremos executar **todos os testes de uma vez** e obter um resumo.
A funcao `full_diagnostics()` do archbox faz exatamente isso.

Adicionalmente, criamos uma funcao customizada que retorna um **DataFrame**
com todos os p-valores para facilitar a comparacao entre modelos.

In [ ]:
# TODO: Crie funcao que executa todos os testes e retorna DataFrame resumo

def diagnostic_summary(model, results, model_name="Model"):
    """Execute todos os testes diagnosticos e retorna DataFrame com p-valores."""
    std_resids = results.resid
    raw_resids = model.endog

    rows = []

    # ARCH-LM
    for lags in [1, 5, 10]:
        result = arch_lm_test(std_resids, lags=lags)
        rows.append({
            'Modelo': model_name,
            'Teste': f'ARCH-LM({lags})',
            'Estatistica': result.statistic,
            'p-valor': result.pvalue,
            'H0': 'Sem efeitos ARCH',
        })

    # Ljung-Box
    for lags in [5, 10, 20]:
        result = ljung_box_squared(std_resids, lags=lags)
        rows.append({
            'Modelo': model_name,
            'Teste': f'Ljung-Box({lags})',
            'Estatistica': result.statistic,
            'p-valor': result.pvalue,
            'H0': 'Sem autocorrelacao em z^2',
        })

    # Sign Bias
    sb = sign_bias_test(raw_resids, std_resids)
    rows.append({
        'Modelo': model_name,
        'Teste': 'Sign Bias',
        'Estatistica': sb.sign_bias[0],
        'p-valor': sb.sign_bias[1],
        'H0': 'Sem assimetria de sinal',
    })
    rows.append({
        'Modelo': model_name,
        'Teste': 'Neg. Size Bias',
        'Estatistica': sb.neg_sign_bias[0],
        'p-valor': sb.neg_sign_bias[1],
        'H0': 'Sem assimetria negativa',
    })
    rows.append({
        'Modelo': model_name,
        'Teste': 'Pos. Size Bias',
        'Estatistica': sb.pos_sign_bias[0],
        'p-valor': sb.pos_sign_bias[1],
        'H0': 'Sem assimetria positiva',
    })
    rows.append({
        'Modelo': model_name,
        'Teste': 'Sign Bias (Joint)',
        'Estatistica': sb.joint[0],
        'p-valor': sb.joint[1],
        'H0': 'Sem assimetria (conjunto)',
    })

    # Nyblom (se scores disponiveis)
    scores = compute_score_matrix(model, results)
    ny = nyblom_test(scores)
    rows.append({
        'Modelo': model_name,
        'Teste': 'Nyblom (Joint)',
        'Estatistica': ny.joint_statistic,
        'p-valor': np.nan,  # Nyblom usa valores criticos, nao p-valor
        'H0': f'Parametros estaveis (cv5%={ny.critical_values_joint[1]:.3f})',
    })

    return pd.DataFrame(rows)

# Comparacao: GARCH simetrico vs EGARCH nos dados assimetricos
df_garch = diagnostic_summary(model_asym, results_asym, "GARCH(1,1)")
df_egarch = diagnostic_summary(model_egarch, results_egarch, "EGARCH(1,1)")

df_all = pd.concat([df_garch, df_egarch], ignore_index=True)

# Formatar e exibir
print("=" * 80)
print("WORKFLOW DIAGNOSTICO: Comparacao GARCH simetrico vs EGARCH assimetrico")
print("Dados: retornos gerados por EGARCH (assimetricos)")
print("=" * 80)
print()

# Pivot para comparacao lado a lado
pivot = df_all.pivot_table(
    index='Teste', columns='Modelo', values='p-valor', aggfunc='first'
)
print(pivot.round(4).to_string())

print("\nDecisao: p-valor > 0.05 → PASS (nao rejeita H0)")
print("O EGARCH deve ter diagnosticos melhores nos dados assimetricos.")

## 9. Usando full_diagnostics()

O archbox fornece a funcao `full_diagnostics()` que executa **todos os testes de uma vez**
e gera um relatorio formatado. Isso e util para uma avaliacao rapida do modelo.

In [ ]:
# TODO: Use full_diagnostics() para gerar relatorio automatico
# Diagnostico completo do modelo GARCH(1,1) nos dados SP500

report = full_diagnostics(results_garch)
print(report.summary())

print("\n\nNota: full_diagnostics() executa ARCH-LM, Sign Bias, Ljung-Box,")
print("Nyblom (se scores disponiveis) e Jarque-Bera automaticamente.")

## 10. Comparacao GARCH simetrico vs assimetrico via diagnosticos

Resumo final: usamos os diagnosticos para **guiar a selecao de modelo**.
Se o GARCH simetrico falha nos testes de assimetria (sign bias), devemos
migrar para um modelo assimetrico (EGARCH, GJR-GARCH).

| Teste | O que detecta | Se falha, considerar... |
|---|---|---|
| ARCH-LM | Efeitos ARCH residuais | Aumentar ordem (p,q) |
| Ljung-Box | Autocorrelacao em $z_t^2$ | Aumentar ordem ou mudar especificacao |
| Sign Bias | Assimetria nao capturada | EGARCH, GJR-GARCH, APARCH |
| Nyblom | Instabilidade parametrica | Regime-switching, rolling window |

In [ ]:
# TODO: Comparacao final — GJR-GARCH tambem nos dados assimetricos

model_gjr = GJRGARCH(returns_asym, p=1, q=1)
results_gjr = model_gjr.fit(disp=False)

df_gjr = diagnostic_summary(model_gjr, results_gjr, "GJR-GARCH(1,1)")

df_final = pd.concat([df_garch, df_egarch, df_gjr], ignore_index=True)

pivot_final = df_final.pivot_table(
    index='Teste', columns='Modelo', values='p-valor', aggfunc='first'
)

print("=" * 80)
print("COMPARACAO FINAL: Diagnosticos para 3 modelos em dados assimetricos")
print("=" * 80)
print()
print(pivot_final.round(4).to_string())

print("\nConclusao:")
print("- GARCH simetrico falha nos testes de sign bias (esperado)")
print("- EGARCH e GJR-GARCH capturam a assimetria corretamente")
print("- Modelos assimetricos produzem diagnosticos superiores")